In [2]:
import os, openai

api_key = os.environ.get("DEEPSEEK_API_KEY")
base_url = "https://api.deepseek.com/"
model = "deepseek-v4-flash"

## 标签生成
+ 流程: 传入一段非结构化文本(附带结构化描述) -> 使用语言模型生成结构化输出（分析输入文本），按照传入的结构描述创建响应
+ 要求：生成包含文本情感对象，添加文本语言的标签
+ 返回：包含情感标签和语言标签的对象

## 提取用例
+ 流程：从文本中提取特定实体，其中实体由结构化描述表示，而非通过语言模型分析文本；使用语言模型扫描文本提取元素列表

In [3]:
# 导入标准配置
from typing import List
from pydantic import BaseModel, Field
from langchain_classic.utils.openai_functions import convert_pydantic_to_openai_tool

In [6]:
class Tagging(BaseModel):
    # 告诉语言模型期望提取的数据结构
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, `neutral`")
    # 记录文本的语言，并指定编码
    language: str = Field(description="language of text (should be ISO 639-1 code)")

In [7]:
convert_pydantic_to_openai_tool(Tagging)

{'type': 'function',
 'function': {'name': 'Tagging',
  'description': 'Tag the piece of text with particular info.',
  'parameters': {'properties': {'sentiment': {'description': 'sentiment of text, should be `pos`, `neg`, `neutral`',
     'type': 'string'},
    'language': {'description': 'language of text (should be ISO 639-1 code)',
     'type': 'string'}},
   'required': ['sentiment', 'language'],
   'type': 'object'}}}

In [8]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI

In [10]:
model = ChatOpenAI(
    model=model,
    api_key=api_key,
    base_url=base_url,
    temperature=0

)

In [11]:
tagging_functions = [convert_pydantic_to_openai_tool(Tagging)]

In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [26]:
model_with_function = model.bind_tools(
    tagging_functions,
    tool_choice="Tagging",
    extra_body={"thinking": {"type": "disabled"}}, # 需要关闭思考模式才能强制使用tool
)

In [27]:
tagging_chain = prompt | model_with_function

In [28]:
tagging_chain.invoke({"input": "I love langchain"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 348, 'total_tokens': 401, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 348}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'd2d84946-8f30-4705-bd2e-bdc13ea4b3bb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0809f-7c0a-7453-afb8-c8b2e7195814-0', tool_calls=[{'name': 'Tagging', 'args': {'sentiment': 'pos', 'language': 'en'}, 'id': 'call_00_P7mU4QyAgQBuRcIyWnmf6295', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 348, 'output_tokens': 53, 'total_tokens': 401, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})